# RISC-V Model Evaluation

Evaluates the trained LoRA adapter on 3 metrics:

| # | Metric | What it measures |
|---|--------|------------------|
| 1 | **Perplexity** | Language model confidence (lower = better) |
| 2 | **Exact Match (EM)** | % of completions that match reference exactly |
| 3 | **BLEU** | n-gram overlap (partial credit for near-misses) |

**Model:** `trained_model/final_model` (LoRA adapter on codegen-350M-multi)  
**Test set:** `dataset/v1/fim/test.jsonl` (real held-out examples from training data)

## 1. Install dependencies

In [10]:
!pip install -q torch transformers peft accelerate sacrebleu


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Imports & config

In [11]:
import json
import math
import re
import random
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import sacrebleu

# ── Paths ──────────────────────────────────────────────────────────────────
ADAPTER_PATH   = "../trained_model/final_model"
BASE_MODEL     = "Salesforce/codegen-350M-multi"
TEST_FILE      = "./dataset/v1/fim/test.jsonl"
OUTPUT_FILE    = "eval_results.json"

# ── FIM tokens (must match training) ───────────────────────────────────────
FIM_PREFIX = "<fim_prefix>"
FIM_SUFFIX = "<fim_suffix>"
FIM_MIDDLE = "<fim_middle>"

# ── Eval config ────────────────────────────────────────────────────────────
NUM_SAMPLES    = 200   # how many examples to eval (set None to use all 5362)
RANDOM_SEED    = 42
MAX_NEW_TOKENS = 60
MAX_INPUT_LEN  = 512

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE, DTYPE = "cuda", torch.float16
elif torch.backends.mps.is_available():
    DEVICE, DTYPE = "mps", torch.float32
else:
    DEVICE, DTYPE = "cpu", torch.float32

print(f"Device: {DEVICE} | Dtype: {DTYPE}")

Device: mps | Dtype: torch.float32


## 3. Load model

In [12]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float32,
    low_cpu_mem_usage=True,
)
base_model.resize_token_embeddings(len(tokenizer))

print("Merging LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH).merge_and_unload()
model = model.to(dtype=DTYPE, device=DEVICE)
model.eval()

print(f"Model ready ({model.num_parameters():,} params)")

Loading tokenizer...
Loading base model...


Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Merging LoRA adapter...
Model ready (354,864,250 params)


## 4. Load test set from JSONL

In [13]:
def parse_fim(text: str) -> dict:
    """Parse a FIM text into prefix / suffix / reference."""
    prefix    = text.split(FIM_SUFFIX)[0].replace(FIM_PREFIX, "")
    suffix    = text.split(FIM_SUFFIX)[1].split(FIM_MIDDLE)[0]
    reference = text.split(FIM_MIDDLE)[1]
    return {"prefix": prefix, "suffix": suffix, "reference": reference}


# Load all examples
all_cases = []
with open(TEST_FILE) as f:
    for line in f:
        all_cases.append(parse_fim(json.loads(line)["text"]))

# Sample a subset for faster eval (change NUM_SAMPLES=None to run all)
if NUM_SAMPLES and NUM_SAMPLES < len(all_cases):
    random.seed(RANDOM_SEED)
    TEST_CASES = random.sample(all_cases, NUM_SAMPLES)
    print(f"Sampled {NUM_SAMPLES} / {len(all_cases)} examples")
else:
    TEST_CASES = all_cases
    print(f"Using all {len(all_cases)} examples")

# Preview one example
ex = TEST_CASES[0]
print(f"\nExample:")
print(f"  prefix    : {ex['prefix'][:80].strip()}")
print(f"  suffix    : {ex['suffix'][:80].strip()}")
print(f"  reference : {ex['reference'][:80].strip()}")

Sampled 200 / 5362 examples

Example:
  prefix    : delta:
    addi sp, sp, -64
    sw ra, 60(sp)
    sw fp, 56(sp)
    addi fp, sp,
  suffix    : addi sp, sp, 4
    div t0, t0, t1
    sw t0, -8(fp)
    j endif2
  reference : mv t0, a0
    sw t0, -8(fp)
    lw t0, -8(fp)
    li t1, 0.0
    sgt t0, t0,


## 5. Generation helper

In [14]:
def generate(prefix: str, suffix: str, max_new_tokens: int = MAX_NEW_TOKENS) -> str:
    prompt = f"{FIM_PREFIX}{prefix}{FIM_SUFFIX}{suffix}{FIM_MIDDLE}"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated   = tokenizer.decode(outputs[0], skip_special_tokens=True)
    prompt_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)

    if generated.startswith(prompt_text):
        return generated[len(prompt_text):].strip()
    return generated.strip()


# Smoke test on first real example
ex = TEST_CASES[0]
out = generate(ex["prefix"], ex["suffix"])
print(f"reference : {ex['reference'][:80].strip()}")
print(f"generated : {out[:80].strip()}")

reference : mv t0, a0
    sw t0, -8(fp)
    lw t0, -8(fp)
    li t1, 0.0
    sgt t0, t0,
generated : mv t0, a0
    sw t0, -12(fp)
    li t0, 0
    addi sp, sp, -4
    sw t0, 0(sp)


## 6. Run all test cases

In [15]:
results = []

for i, tc in enumerate(TEST_CASES):
    generated = generate(tc["prefix"], tc["suffix"])
    results.append({
        "reference": tc["reference"],
        "generated": generated,
    })
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(TEST_CASES)} done")

print(f"\n{len(results)} cases generated")

  20/200 done
  40/200 done
  60/200 done
  80/200 done
  100/200 done
  120/200 done
  140/200 done
  160/200 done
  180/200 done
  200/200 done

200 cases generated


## 7. Compute metrics

In [20]:
def normalise(text: str) -> str:
    text = re.sub(r"#.*", "", text)        # strip comments
    text = re.sub(r"[ \t]+", " ", text)    # collapse whitespace
    return text.strip().lower()


# ── Exact Match ────────────────────────────────────────────────────────────
# Strict: generated must equal reference exactly (after normalisation)
exact_matches = [
    normalise(r["reference"]) == normalise(r["generated"])
    for r in results
]
exact_match_pct = sum(exact_matches) / len(exact_matches) * 100

# ── Prefix Match ───────────────────────────────────────────────────────────
# Lenient: generated starts with the reference (model is correct but kept going)
prefix_matches = [
    normalise(r["generated"]).startswith(normalise(r["reference"]))
    for r in results
]
prefix_match_pct = sum(prefix_matches) / len(prefix_matches) * 100

# ── BLEU ───────────────────────────────────────────────────────────────────
refs = [[r["reference"]] for r in results]
hyps = [r["generated"]   for r in results]
bleu_score = sacrebleu.corpus_bleu(hyps, list(map(list, zip(*refs)))).score

# ── Print ──────────────────────────────────────────────────────────────────
print("=" * 52)
print("EVALUATION RESULTS")
print("=" * 52)
print(f"  Test cases    : {len(results)}")
print(f"  Exact Match   : {exact_match_pct:.1f}%  (generated == reference)")
print(f"  Prefix Match  : {prefix_match_pct:.1f}%  (generated starts with reference)")
print(f"  BLEU          : {bleu_score:.1f}")
print("=" * 52)
print()
print("Prefix Match is the more meaningful metric here.")
print("EM=0 but PM>0 means correct code but model over-generates.")

# Show interesting cases
close = [r for r, em, pm in zip(results, exact_matches, prefix_matches) if pm and not em]
wrong = [r for r, pm   in zip(results, prefix_matches) if not pm]

print(f"\n-- Correct but over-generated ({len(close)}) --")
for r in close[:3]:
    print(f"  ref: {r['reference'][:70].strip()}")
    print(f"  gen: {r['generated'][:70].strip()}")
    print()

print(f"-- Wrong ({len(wrong)}) --")
for r in wrong[:3]:
    print(f"  ref: {r['reference'][:70].strip()}")
    print(f"  gen: {r['generated'][:70].strip()}")
    print()

EVALUATION RESULTS
  Test cases    : 200
  Exact Match   : 0.0%  (generated == reference)
  Prefix Match  : 22.5%  (generated starts with reference)
  BLEU          : 47.0

Prefix Match is the more meaningful metric here.
EM=0 but PM>0 means correct code but model over-generates.

-- Correct but over-generated (45) --
  ref: j Fibonacci_end0
    j endif4
  gen: j Fibonacci_end0
    j endif4
    j endif5
    j endif6
    j endif8

  ref: sw t0, -12(fp)
    j endif4
  gen: sw t0, -12(fp)
    j endif4 TheNitromeFan    li t0, 0
    sw t0, -8(fp

  ref: mv t0, a0
    sw t0, -4(fp)
  gen: mv t0, a0
    sw t0, -4(fp)
    lw a0, -4(fp)
    call str
    mv t0,

-- Wrong (155) --
  ref: mv t0, a0
    sw t0, -8(fp)
    lw t0, -8(fp)
    li t1, 0.0
    s
  gen: mv t0, a0
    sw t0, -12(fp)
    li t0, 0
    addi sp, sp, -4
    sw t

  ref: addi sp, sp, 4
    add t0, t0, t1
    addi sp, sp, -4
    sw t0, 0
  gen: addi sp, sp, 4
    add t0, t0, t1
    addi sp, sp, -4
    sw t0, 0(sp)

  ref: call num

## 8. Perplexity on test set

In [19]:
def compute_perplexity(prefix: str, suffix: str, reference: str):
    """
    Perplexity of the model on the reference tokens only (prompt masked).
    Returns None if the prompt fills the whole context window (no room for reference).
    """
    prompt = f"{FIM_PREFIX}{prefix}{FIM_SUFFIX}{suffix}{FIM_MIDDLE}"
    full   = prompt + reference

    enc_prompt = tokenizer(prompt, return_tensors="pt")["input_ids"]
    enc_full   = tokenizer(full, return_tensors="pt", truncation=True, max_length=MAX_INPUT_LEN)["input_ids"]

    prompt_len   = enc_prompt.shape[1]
    full_len     = enc_full.shape[1]
    ref_tok_len  = full_len - prompt_len

    # If prompt already fills the window, no reference tokens remain → skip
    if ref_tok_len <= 0:
        return None

    enc_full = enc_full.to(DEVICE)
    labels   = enc_full.clone()
    labels[0, :prompt_len] = -100   # mask prompt tokens

    with torch.no_grad():
        loss = model(enc_full, labels=labels).loss

    val = math.exp(loss.item())
    return val if math.isfinite(val) else None


perplexities = []
skipped = 0
for tc in TEST_CASES:
    ppl = compute_perplexity(tc["prefix"], tc["suffix"], tc["reference"])
    if ppl is not None:
        perplexities.append(ppl)
    else:
        skipped += 1

avg_ppl = sum(perplexities) / len(perplexities)
print(f"Average perplexity : {avg_ppl:.3f}  (over {len(perplexities)} examples, {skipped} skipped — prompt too long)")
print(f"Validation PPL during training : ~2.49")
print(f"(Lower = better. Close to training PPL = good generalisation)")

Average perplexity : 1.491  (over 199 examples, 1 skipped — prompt too long)
Validation PPL during training : ~2.49
(Lower = better. Close to training PPL = good generalisation)


## 9. Save results

In [21]:
summary = {
    "model"       : BASE_MODEL,
    "adapter"     : ADAPTER_PATH,
    "dataset"     : TEST_FILE,
    "num_samples" : len(results),
    "metrics": {
        "exact_match_pct"  : round(exact_match_pct, 2),
        "prefix_match_pct" : round(prefix_match_pct, 2),
        "bleu"             : round(bleu_score, 2),
        "avg_perplexity"   : round(avg_ppl, 4),
        "ppl_sample_size"  : len(perplexities),
    },
}

with open(OUTPUT_FILE, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Results saved to {OUTPUT_FILE}")
print(json.dumps(summary["metrics"], indent=2))

Results saved to eval_results.json
{
  "exact_match_pct": 0.0,
  "prefix_match_pct": 22.5,
  "bleu": 47.03,
  "avg_perplexity": 1.4912,
  "ppl_sample_size": 199
}
